### Phase 2 — Sequence Window Generation + GRU Model

In [53]:
import numpy as np
import pandas as pd

train_df = pd.read_csv(
    '../data/processed/train_final_processed.csv'
)

In [54]:
#Define Final Feature Set
selected_features = [
    'sensor_2',
    'sensor_3',
    'sensor_4',
    'sensor_6',
    'sensor_7',
    'sensor_8',
    'sensor_9',
    'sensor_11',
    'sensor_12',
    'sensor_13',
    'sensor_14',
    'sensor_15',
    'sensor_17',
    'sensor_20',
    'sensor_21'
]

In [55]:
#Add RUL Clipping
train_df['RUL'] = train_df['RUL'].clip(upper=125)

In [56]:
sequence_length = 30 # Hyper Parameter

In [57]:
# Create Sequence Generation Function
def create_sequences(df, sequence_length, feature_cols):

    X = []
    y = []

    engine_ids = df['engine_id'].unique()

    for engine_id in engine_ids:

        engine_data = df[df['engine_id'] == engine_id]

        feature_array = engine_data[feature_cols].values
        rul_array = engine_data['RUL'].values

        for i in range(len(engine_data) - sequence_length):

            X.append(
                feature_array[i:i + sequence_length]
            )

            y.append(
                rul_array[i + sequence_length]
            )

    return np.array(X), np.array(y)

In [58]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_df[selected_features] = scaler.fit_transform(
    train_df[selected_features]
)

In [59]:
# Generate Sequences
X, y = create_sequences(
    train_df,
    sequence_length,
    selected_features
)

In [60]:
print("X shape:", X.shape)
print("y shape:", y.shape)
''' 
30 = timesteps
15 = features 
'''

X shape: (17631, 30, 15)
y shape: (17631,)


' \n30 = timesteps\n15 = features \n'

In [61]:
#Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [62]:
print(X_train.shape)
print(X_val.shape)


(14104, 30, 15)
(3527, 30, 15)


In [63]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

In [19]:
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([

    GRU(
        128,
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2])
    ),

    Dropout(0.3),

    GRU(64),

    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(16, activation='relu'),

    Dense(1)
])

C:\Projects\VALIARIA_Pred_maint\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [20]:
#Compile Model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru (GRU)                            │ (None, 30, 128)             │          55,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 30, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_1 (GRU)                          │ (None, 64)                  │          37,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 95,553 (373.25 KB)

 Trainable params: 95,553 (373.25 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
#We are not using this model 
#Train Model
history = model.fit(

    X_train,
    y_train,

    epochs=50,
    batch_size=64,

    validation_data=(X_val, y_val),

    callbacks=[early_stop]
)

In [ ]:
#Evaluate Model
from sklearn.metrics import mean_squared_error
predictions = model.predict(X_val)

rmse = np.sqrt(
    mean_squared_error(y_val, predictions)
)

print("Improved RMSE:", rmse)

In [ ]:
# Visualize Training Curves
plt.figure(figsize=(10,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')

plt.legend()
plt.show()

In [ ]:
# Then Check Prediction Quality
plt.figure(figsize=(10,5))

plt.plot(y_val[:200], label='Actual RUL')
plt.plot(predictions[:200], label='Predicted RUL')

plt.xlabel('Sample')
plt.ylabel('RUL')

plt.title('Actual vs Predicted RUL')

plt.legend()
plt.show()

### ONE more optimization round

## Bidirectional GRU + Learning Rate Reduction

In [64]:
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [65]:
model = Sequential([

    Bidirectional(
        GRU(
            128,
            return_sequences=True
        ),
        input_shape=(X_train.shape[1], X_train.shape[2])
    ),

    Dropout(0.3),

    Bidirectional(
        GRU(64)
    ),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(32, activation='relu'),

    Dense(1)
])

C:\Projects\VALIARIA_Pred_maint\venv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [66]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [67]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

In [68]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [69]:
history = model.fit(

    X_train,
    y_train,

    epochs=50,
    batch_size=64,

    validation_data=(X_val, y_val),

    callbacks=[
        early_stop,
        reduce_lr
    ]
)

Epoch 1/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - loss: 3742.4336 - mae: 50.7569 - val_loss: 1510.0907 - val_mae: 34.7723 - learning_rate: 0.0010
Epoch 2/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 823.0768 - mae: 23.6813 - val_loss: 254.9599 - val_mae: 11.8822 - learning_rate: 0.0010
Epoch 3/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 268.2935 - mae: 12.5312 - val_loss: 196.3440 - val_mae: 10.7223 - learning_rate: 0.0010
Epoch 4/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 236.4038 - mae: 11.6396 - val_loss: 181.9859 - val_mae: 9.7822 - learning_rate: 0.0010
Epoch 5/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 222.8945 - mae: 11.2228 - val_loss: 190.4095 - val_mae: 9.9174 - learning_rate: 0.0010
Epoch 6/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 216.9922 - mae: 11.1090 - val_loss: 180.7828 - val_mae: 9.8557 - learning_rate: 0.0010
Epoch 7/50
221/221 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 208.3394 - mae: 10.7203 - val_loss: 188

In [70]:
from sklearn.metrics import mean_squared_error


In [71]:
predictions = model.predict(X_val)

rmse = np.sqrt(
    mean_squared_error(y_val, predictions)
)

print("Final Optimized RMSE:", rmse)

111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
Final Optimized RMSE: 13.0632540106468


In [72]:
model.save("../models/bigru_rul_model.h5")

In [73]:
import joblib

joblib.dump(
    scaler,
    "../models/minmax_scaler.pkl"
)

['../models/minmax_scaler.pkl']